#**Preprocess manually**


In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("jopacc_system_month_november_2022_to_june_2026.csv")
df.head()

,report_month,report_year,report_quarter,system,users_total,transactions_total,transaction_value_jod,user_jordanian_count,user_non_jordanian_count,user_male_count,...,ach_usd_transaction_value,ach_eur_transaction_count,ach_eur_transaction_value,ach_gbp_transaction_count,ach_gbp_transaction_value,returned_cheque_count,returned_cheque_value_jod,returned_cheque_count_rate_pct,returned_cheque_value_rate_pct,activity_level
0,2022-11,2022,Q4,CliQ,523000,985000,175000000,494200,24100,342500,...,0,0,0,0,0,0,0,0.0,0.0,Low Activity
1,2022-11,2022,Q4,JoMoPay,2020000,1260000,155000000,1760000,250000,1230000,...,0,0,0,0,0,0,0,0.0,0.0,Low Activity
2,2022-11,2022,Q4,eFAWATEERcom,3580000,3630000,848000000,0,0,0,...,0,0,0,0,0,0,0,0.0,0.0,Low Activity
3,2022-12,2022,Q4,CliQ,561000,1180000,209000000,529800,26300,365000,...,0,0,0,0,0,0,0,0.0,0.0,Low Activity
4,2022-12,2022,Q4,JoMoPay,2050000,1510000,188000000,1780000,250000,1230000,...,0,0,0,0,0,0,0,0.0,0.0,Low Activity


**Prepare the month column**

In [2]:
df["report_month"] = pd.to_datetime(df["report_month"])

df["report_month_number"] = (df["report_month"].dt.month)

df[
    [
        "report_month",
        "report_month_number",
        "report_year"
    ]
].head()

,report_month,report_month_number,report_year
0,2022-11-01,11,2022
1,2022-11-01,11,2022
2,2022-11-01,11,2022
3,2022-12-01,12,2022
4,2022-12-01,12,2022


**Numeric (magnitude)**

In [3]:
num_cols = [
    "report_month_number",
    "report_year",
    "transactions_total",
    "users_total",
    "customer_legal_entity_count",
    "wallet_commercial_count",
    "tx_money_transfers_count",
    "ef_digital_payment_count",
    "ach_jod_transaction_count",
    "returned_cheque_count"
]


**Context (for locality)**

In [4]:
cat_cols = ["system"]

In [5]:
anomaly_data = df[num_cols + cat_cols].copy()

**One-hot encode categorical features**

In [6]:
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
X_cat = ohe.fit_transform(df[cat_cols])


**Scale numeric features (CRITICAL)**

In [7]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_num_scaled = scaler.fit_transform(df[num_cols])


**Final feature matrix**

In [8]:
import numpy as np

X = np.hstack([X_num_scaled, X_cat])
X.shape

(214, 15)

In [9]:
import pickle

cat_options = {
    col: list(cats)
    for col, cats in zip(cat_cols, ohe.categories_)
}

cat_options

{'system': ['ACH', 'CliQ', 'ECCU', 'JoMoPay', 'eFAWATEERcom']}

In [10]:
pickle.dump(scaler, open("models/anomaly_scaler.pkl", "wb"))

In [11]:
import pickle
pickle.dump(ohe, open("models/onehot_encoder.pkl", "wb"))
pickle.dump(num_cols, open("models/num_cols.pkl", "wb"))
pickle.dump(cat_cols, open("models/cat_cols.pkl", "wb"))


#**Isolation Forest (global anomalies)**

What Isolation Forest actually does (short intuition)

* It randomly splits features

* Anomalies get isolated in fewer splits

* Shorter path → more anomalous

So:

* Lower depth = anomaly

* Higher depth = normal

**Basic model**

In [12]:
from sklearn.ensemble import IsolationForest

iso = IsolationForest(
    contamination=0.03,  
    n_estimators=300,    
    max_samples="auto",
    max_features=1.0, 
    random_state=42
)

iso.fit(X)
df["score_iso"] = -iso.decision_function(X)
df.head()

,report_month,report_year,report_quarter,system,users_total,transactions_total,transaction_value_jod,user_jordanian_count,user_non_jordanian_count,user_male_count,...,ach_eur_transaction_value,ach_gbp_transaction_count,ach_gbp_transaction_value,returned_cheque_count,returned_cheque_value_jod,returned_cheque_count_rate_pct,returned_cheque_value_rate_pct,activity_level,report_month_number,score_iso
0,2022-11-01,2022,Q4,CliQ,523000,985000,175000000,494200,24100,342500,...,0,0,0,0,0,0.0,0.0,Low Activity,11,-0.026678
1,2022-11-01,2022,Q4,JoMoPay,2020000,1260000,155000000,1760000,250000,1230000,...,0,0,0,0,0,0.0,0.0,Low Activity,11,-0.031491
2,2022-11-01,2022,Q4,eFAWATEERcom,3580000,3630000,848000000,0,0,0,...,0,0,0,0,0,0.0,0.0,Low Activity,11,-0.028781
3,2022-12-01,2022,Q4,CliQ,561000,1180000,209000000,529800,26300,365000,...,0,0,0,0,0,0.0,0.0,Low Activity,12,-0.024385
4,2022-12-01,2022,Q4,JoMoPay,2050000,1510000,188000000,1780000,250000,1230000,...,0,0,0,0,0,0.0,0.0,Low Activity,12,-0.035692


**Convert scores → anomaly labels**

In [13]:
threshold = np.percentile(df["score_iso"], 97)
df["anomaly_iso"] = (df["score_iso"] >= threshold).astype(int)
df.head()

,report_month,report_year,report_quarter,system,users_total,transactions_total,transaction_value_jod,user_jordanian_count,user_non_jordanian_count,user_male_count,...,ach_gbp_transaction_count,ach_gbp_transaction_value,returned_cheque_count,returned_cheque_value_jod,returned_cheque_count_rate_pct,returned_cheque_value_rate_pct,activity_level,report_month_number,score_iso,anomaly_iso
0,2022-11-01,2022,Q4,CliQ,523000,985000,175000000,494200,24100,342500,...,0,0,0,0,0.0,0.0,Low Activity,11,-0.026678,0
1,2022-11-01,2022,Q4,JoMoPay,2020000,1260000,155000000,1760000,250000,1230000,...,0,0,0,0,0.0,0.0,Low Activity,11,-0.031491,0
2,2022-11-01,2022,Q4,eFAWATEERcom,3580000,3630000,848000000,0,0,0,...,0,0,0,0,0.0,0.0,Low Activity,11,-0.028781,0
3,2022-12-01,2022,Q4,CliQ,561000,1180000,209000000,529800,26300,365000,...,0,0,0,0,0.0,0.0,Low Activity,12,-0.024385,0
4,2022-12-01,2022,Q4,JoMoPay,2050000,1510000,188000000,1780000,250000,1230000,...,0,0,0,0,0.0,0.0,Low Activity,12,-0.035692,0


In [14]:
print("Isolation Forest results:")

print(df["anomaly_iso"].value_counts())

print("\nIsolation Forest anomaly rate:",f"{df['anomaly_iso'].mean():.2%}")

Isolation Forest results:
anomaly_iso
0    207
1      7
Name: count, dtype: int64

Isolation Forest anomaly rate: 3.27%


In [15]:
iso_anomalies = df[df["anomaly_iso"] == 1].sort_values(by="score_iso",ascending=False)

iso_anomalies[num_cols+ cat_cols+ ["score_iso","anomaly_iso"]]

,report_month_number,report_year,transactions_total,users_total,customer_legal_entity_count,wallet_commercial_count,tx_money_transfers_count,ef_digital_payment_count,ach_jod_transaction_count,returned_cheque_count,system,score_iso,anomaly_iso
212,6,2026,14790000,3000000,0,7600,12290000,0,0,0,JoMoPay,0.029336,1
210,6,2026,20960000,2310000,56050,0,16720000,0,0,0,CliQ,0.027851,1
205,5,2026,21430000,2280000,54700,0,17230000,0,0,0,CliQ,0.022095,1
207,5,2026,14980000,2970000,0,7401,12510000,0,0,0,JoMoPay,0.017694,1
179,12,2025,1200000,0,0,0,0,0,1190000,0,ACH,0.005408,1
185,1,2026,17700000,2160000,49400,0,14290000,0,0,0,CliQ,0.003080,1
180,12,2025,17670000,2130000,48300,0,14250000,0,0,0,CliQ,0.000264,1


**Evaluation Method  Synthetic anomaly injection (BEST PRACTICE)**

In [16]:
#Create fake anomalies
df_synth = df.copy()
idx = df_synth.sample(frac=0.03, random_state=42).index

df_synth.loc[idx, "transactions_total"] *= 5
df_synth.loc[idx, "users_total"] *= 4
df_synth.loc[idx, "returned_cheque_count"] = (df["returned_cheque_count"].max()* 3)
df_synth["is_synthetic"] = 0
df_synth.loc[idx, "is_synthetic"] = 1
df_synth.loc[idx,num_cols+ cat_cols+ ["is_synthetic"]]
df_synth.head()

,report_month,report_year,report_quarter,system,users_total,transactions_total,transaction_value_jod,user_jordanian_count,user_non_jordanian_count,user_male_count,...,ach_gbp_transaction_value,returned_cheque_count,returned_cheque_value_jod,returned_cheque_count_rate_pct,returned_cheque_value_rate_pct,activity_level,report_month_number,score_iso,anomaly_iso,is_synthetic
0,2022-11-01,2022,Q4,CliQ,523000,985000,175000000,494200,24100,342500,...,0,0,0,0.0,0.0,Low Activity,11,-0.026678,0,0
1,2022-11-01,2022,Q4,JoMoPay,2020000,1260000,155000000,1760000,250000,1230000,...,0,0,0,0.0,0.0,Low Activity,11,-0.031491,0,0
2,2022-11-01,2022,Q4,eFAWATEERcom,3580000,3630000,848000000,0,0,0,...,0,0,0,0.0,0.0,Low Activity,11,-0.028781,0,0
3,2022-12-01,2022,Q4,CliQ,561000,1180000,209000000,529800,26300,365000,...,0,0,0,0.0,0.0,Low Activity,12,-0.024385,0,0
4,2022-12-01,2022,Q4,JoMoPay,2050000,1510000,188000000,1780000,250000,1230000,...,0,0,0,0.0,0.0,Low Activity,12,-0.035692,0,0


**Re-score and evaluate separation:**

In [17]:
from sklearn.metrics import roc_auc_score

# numeric features → scale using SAME scaler
X_num_synth = scaler.transform(df_synth[num_cols])

# categorical → one-hot encode & ALIGN columns
X_cat_synth = ohe.transform(df_synth[cat_cols])
X_synth = np.hstack([X_num_synth, X_cat_synth])


scores = -iso.decision_function(X_synth)

iso_roc_auc = roc_auc_score(df_synth["is_synthetic"], scores)

print("Isolation Forest ROC-AUC:", round(iso_roc_auc, 4))
#  AUC > 0.85 → strong detector
# 0.75–0.85  → good
# 0.65–0.75  → weak detector
#  ~0.50  →   broken evaluation

Isolation Forest ROC-AUC: 0.9647


#**One-Class SVM)**

**Learn the smallest region in feature space that contains most of the data.**

In [18]:
from sklearn.svm import OneClassSVM

ocsvm = OneClassSVM(
    kernel="rbf",
    nu=0.03,
    gamma="scale",
)
ocsvm.fit(X)


,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm.If none is given, 'rbf' will be used. If a callable is given it isused to precompute the kernel matrix.",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"nu nu: float, default=0.5An upper bound on the fraction of trainingerrors and a lower bound of the fraction of supportvectors. Should be in the interval (0, 1]. By default 0.5will be taken.",0.03
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide `.",True
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False
,"max_iter max_iter: int, default=-1Hard limit on iterations within solver, or -1 for no limit.",-1


In [19]:
# decision_function:
#   positive → inside boundary (normal)
#   negative → outside (anomaly)

df["score_ocsvm"] = -ocsvm.decision_function(X)

# binary labels
df["anomaly_ocsvm"] = (ocsvm.predict(X) == -1).astype(int)
df.head()

,report_month,report_year,report_quarter,system,users_total,transactions_total,transaction_value_jod,user_jordanian_count,user_non_jordanian_count,user_male_count,...,returned_cheque_count,returned_cheque_value_jod,returned_cheque_count_rate_pct,returned_cheque_value_rate_pct,activity_level,report_month_number,score_iso,anomaly_iso,score_ocsvm,anomaly_ocsvm
0,2022-11-01,2022,Q4,CliQ,523000,985000,175000000,494200,24100,342500,...,0,0,0.0,0.0,Low Activity,11,-0.026678,0,-0.025890,0
1,2022-11-01,2022,Q4,JoMoPay,2020000,1260000,155000000,1760000,250000,1230000,...,0,0,0.0,0.0,Low Activity,11,-0.031491,0,-0.034993,0
2,2022-11-01,2022,Q4,eFAWATEERcom,3580000,3630000,848000000,0,0,0,...,0,0,0.0,0.0,Low Activity,11,-0.028781,0,-0.033441,0
3,2022-12-01,2022,Q4,CliQ,561000,1180000,209000000,529800,26300,365000,...,0,0,0.0,0.0,Low Activity,12,-0.024385,0,-0.000389,0
4,2022-12-01,2022,Q4,JoMoPay,2050000,1510000,188000000,1780000,250000,1230000,...,0,0,0.0,0.0,Low Activity,12,-0.035692,0,-0.000008,0


In [20]:
print("One-Class SVM results:")

print(df["anomaly_ocsvm"].value_counts())

print("\nOne-Class SVM anomaly rate:", f"{df['anomaly_ocsvm'].mean():.2%}")

One-Class SVM results:
anomaly_ocsvm
0    201
1     13
Name: count, dtype: int64

One-Class SVM anomaly rate: 6.07%


In [21]:
ocsvm_anomalies = df[
    df["anomaly_ocsvm"] == 1
].sort_values(
    by="score_ocsvm",
    ascending=False
)

ocsvm_anomalies[
    num_cols
    + cat_cols
    + [
        "score_ocsvm",
        "anomaly_ocsvm"
    ]
]

,report_month_number,report_year,transactions_total,users_total,customer_legal_entity_count,wallet_commercial_count,tx_money_transfers_count,ef_digital_payment_count,ach_jod_transaction_count,returned_cheque_count,system,score_ocsvm,anomaly_ocsvm
180,12,2025,17670000,2130000,48300,0,14250000,0,0,0,CliQ,0.000472,1
185,1,2026,17700000,2160000,49400,0,14290000,0,0,0,CliQ,0.000447,1
7,1,2023,1310000,592000,5500,0,1219000,0,0,0,CliQ,0.000405,1
121,12,2024,670000,0,0,0,0,0,0,23800,ECCU,0.000348,1
205,5,2026,21430000,2280000,54700,0,17230000,0,0,0,CliQ,0.000270,1
120,12,2024,10000000,1670000,30400,0,8570000,0,0,0,CliQ,0.000259,1
188,1,2026,6530000,5090000,0,0,0,5280000,0,0,eFAWATEERcom,0.000213,1
182,12,2025,10930000,2810000,0,6280,9240000,0,0,0,JoMoPay,0.000185,1
187,1,2026,11830000,2850000,0,6436,10010000,0,0,0,JoMoPay,0.000149,1
196,3,2026,567000,0,0,0,0,0,0,20100,ECCU,0.000114,1


In [22]:
for g in ["scale", 0.01, 0.1]:
    oc = OneClassSVM(nu=0.03, gamma=g)
    oc.fit(X)

    scores_oc= -oc.decision_function(X_synth)

    print(
        "gamma:", g,
        "roc:", roc_auc_score(df_synth["is_synthetic"], scores_oc)
    )

gamma: scale roc: 1.0
gamma: 0.01 roc: 1.0
gamma: 0.1 roc: 1.0


#**iso_roc_auc VS ocsvm_roc_auc)**

In [23]:
ocsvm_roc_auc=roc_auc_score(df_synth["is_synthetic"], scores_oc)
results = pd.DataFrame({"Model": ["Isolation Forest","One-Class SVM"], "ROC-AUC": [iso_roc_auc,ocsvm_roc_auc],
                        "Detected Anomalies": [int(df["anomaly_iso"].sum()), int(df["anomaly_ocsvm"].sum())],
                        "Anomaly Rate": [df["anomaly_iso"].mean(), df["anomaly_ocsvm"].mean()]
                        })

print("Anomaly Model Comparison:")

print(results.to_string(index=False))

Anomaly Model Comparison:
           Model  ROC-AUC  Detected Anomalies  Anomaly Rate
Isolation Forest 0.964744                   7      0.032710
   One-Class SVM 1.000000                  13      0.060748


In [24]:
if iso_roc_auc >= ocsvm_roc_auc:
    best_model_name = "Isolation Forest"
    best_anomaly_model = iso

else:
    best_model_name = "One-Class SVM"
    best_anomaly_model = ocsvm

print("\nBest anomaly model:",best_model_name)



Best anomaly model: One-Class SVM


In [25]:
import pickle

with open("models/best_anomaly_model.pkl", "wb") as f:
    pickle.dump(
        best_anomaly_model, f
    )


print(
    "\nBest anomaly model saved to: "
    "models/best_anomaly_model.pkl"
)


Best anomaly model saved to: models/best_anomaly_model.pkl


In [26]:
with open(
    "models/scaler.pkl",
    "wb"
) as file:

    pickle.dump(
        scaler,
        file
    )